# **AI Recruitment Assistant**
### **Powered by Groq Llama-3.3-70B, LangChain, and Advanced RAG (Hybrid Search)**

This system follows the **CRISP-DM** (Cross-Industry Standard Process for Data Mining) methodology to ensure a standardized and professional workflow.

## **Phase 1: Business Understanding**

### **1.1 Background**
Manual recruitment is often inefficient and prone to human bias. HR departments need tools for objective and fast CV screening.

### **1.2 Business Objectives**
- Automate CV screening against Job Descriptions (JD).
- Provide objective and transparent match scores.
- Offer automated initial interview features to save recruiter time.

### **1.3 Data Mining Goals**
- Perform semantic matching between CV and JD using Hybrid Search (BM25 + Dense).
- Generate relevant interview questions based on candidate qualifications.
- Evaluate candidate answers systematically with bias checks.

## **Phase 2: Data Understanding**

### **2.1 Data Sources**
1. **Resume (CV)**: PDF documents containing candidate professional profiles.
2. **Job Description (JD)**: Text descriptions of job roles and requirements.

## **Phase 3: Data Preparation**

In [1]:
# 📥 Install Dependencies
!pip install -q groq langchain pypdf gradio faiss-cpu sentence-transformers langchain-experimental rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
#import library
import os
import re
import json
import logging
from typing import List, Tuple, Optional, Any

import gradio as gr
from pydantic import BaseModel, Field
from groq import Groq

# LangChain & RAG Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ⚙️ Configuration & Constants
DEFAULT_GROQ_API_KEY = ""
GROQ_API_KEY = os.getenv("GROQ_API_KEY", DEFAULT_GROQ_API_KEY)
MODEL_NAME = "llama-3.3-70b-versatile"
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Global model instances
print("Loading embedding and reranker models... This may take a minute.")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
reranker = CrossEncoder(RERANKER_MODEL)
client = Groq(api_key=GROQ_API_KEY)
print("Models loaded successfully!")

Loading embedding and reranker models... This may take a minute.


/tmp/ipykernel_2288/329908238.py:33: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Models loaded successfully!


In [3]:
class DocumentProcessor:
    """Handles PDF loading and text preprocessing."""

    @staticmethod
    def load_cv(file_path: str, max_chars: int = 16000) -> str:
        """Extract text from PDF and truncate to fit context limits."""
        try:
            loader = PyPDFLoader(file_path)
            docs = loader.load()

            full_context = ""
            for doc in docs:
                if len(full_context) + len(doc.page_content) <= max_chars:
                    full_context += doc.page_content + "\n"
                else:
                    break
            return full_context.strip()
        except Exception as e:
            logger.error(f"Error loading CV: {e}")
            return ""

## **Phase 4: Modeling**

In [4]:
# Structured Data Models
class ResumeAnalysis(BaseModel):
    match_score: int = Field(..., ge=0, le=100)
    key_strengths: List[str]
    missing_skills: List[str]
    final_decision: str
    message: str
    detailed_analysis: str

class InterviewQuestions(BaseModel):
    questions: List[str]

class QuestionScore(BaseModel):
    question: str
    score: int = Field(..., ge=0, le=10)
    feedback: str

class EvaluationResult(BaseModel):
    question_scores: List[QuestionScore]
    average_score: float
    accuracy_matching: int = Field(..., ge=0, le=100)
    consistency_score: int = Field(..., ge=0, le=100)
    bias_check: str
    decision: str
    overall_feedback: str

In [5]:
class RAGEngine:
    """Manages Hybrid Search (BM25 + Vector) and Reranking."""

    def __init__(self, embeddings, reranker):
        self.embeddings = embeddings
        self.reranker = reranker
        self.priority_terms = ["skill", "requirement", "technolog", "experience", "kualifikasi"]

    def get_relevant_context(self, jd_text: str, cv_context: str, k: int = 3, top_n: int = 15) -> str:
        if len(jd_text) < 1000:
            return jd_text

        try:
            # 1. Semantic Chunking
            semantic_splitter = SemanticChunker(self.embeddings)
            jd_chunks = semantic_splitter.split_text(jd_text)

            # 2. Sparse Search (BM25)
            def tokenize(text):
                return re.sub(r'[^\w\s]', '', text.lower()).split()

            tokenized_chunks = [tokenize(chunk) for chunk in jd_chunks]
            bm25 = BM25Okapi(tokenized_chunks)
            bm25_hits = bm25.get_top_n(tokenize(cv_context), jd_chunks, n=top_n)

            # 3. Dense Search (Vector)
            vectorstore = FAISS.from_texts(jd_chunks, self.embeddings)
            dense_hits = [doc.page_content for doc in vectorstore.similarity_search(cv_context, k=top_n)]

            # 4. Pool & Deduplicate
            all_candidates = list(set(bm25_hits + dense_hits))

            # 5. Reranking (Cross-Encoder)
            pairs = [[cv_context, cand] for cand in all_candidates]
            scores = self.reranker.predict(pairs)

            # 6. Priority Boosting & Final Selection
            scored_candidates = []
            for score, cand in zip(scores, all_candidates):
                if any(term in cand.lower() for term in self.priority_terms):
                    score += 0.3
                scored_candidates.append((score, cand))

            final_candidates = sorted(scored_candidates, key=lambda x: x[0], reverse=True)
            top_results = [cand for _, cand in final_candidates[:k]]

            return "\n\n".join(top_results)
        except Exception as e:
            logger.warning(f"Hybrid Search Warning: {e}")
            return jd_text

In [6]:
class RecruitmentAI:
    """Orchestrates the AI recruitment logic."""

    def __init__(self, client, rag_engine):
        self.client = client
        self.rag = rag_engine

    def _call_llm(self, system_prompt: str, user_prompt: str, response_model: Any) -> Any:
        try:
            response = self.client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"}
            )
            raw_content = response.choices[0].message.content
            parsed_json = json.loads(raw_content)

            for key in [response_model.__name__, "analysis", "data", "evaluation"]:
                if key in parsed_json and isinstance(parsed_json[key], dict):
                    parsed_json = parsed_json[key]
                    break

            return response_model.model_validate(parsed_json)
        except Exception as e:
            logger.error(f"LLM Call Error: {e}")
            raise e

    def analyze_resume(self, cv_file_path: str, jd_text: str) -> Tuple[str, List[str], int]:
        try:
            cv_context = DocumentProcessor.load_cv(cv_file_path)
            relevant_jd = self.rag.get_relevant_context(jd_text, cv_context)

            system_prompt = "You are an expert HR Auditor. Return only raw JSON matching the requested schema."
            user_prompt = f"""Analyze this resume against the Job Description.
            Resume Content: {cv_context}
            Relevant JD: {relevant_jd}

            Return a JSON object matching this schema: {ResumeAnalysis.model_json_schema()}"""

            analysis = self._call_llm(system_prompt, user_prompt, ResumeAnalysis)

            result_md = f"""### 📊 Match Score: {analysis.match_score}/100
**Decision: {analysis.final_decision}**

**Key Strengths:**
{chr(10).join([f'- {s}' for s in analysis.key_strengths])}

**Missing Skills:**
{chr(10).join([f'- {s}' for s in analysis.missing_skills])}

**Detailed Analysis:**
{analysis.detailed_analysis}"""

            questions = []
            if analysis.match_score >= 60:
                questions = self.generate_questions(cv_context, relevant_jd)

            return result_md, questions, analysis.match_score
        except Exception as e:
            return f"❌ Error analyzing resume: {str(e)}", [], 0

    def generate_questions(self, cv_context: str, jd_context: str) -> List[str]:
        try:
            system_prompt = "You are a technical interviewer. Output JSON only."
            user_prompt = f"""Generate 3-5 technical interview questions based on the CV and JD provided.
            CV: {cv_context}
            JD: {jd_context}

            Return a JSON object matching this schema: {InterviewQuestions.model_json_schema()}"""

            data = self._call_llm(system_prompt, user_prompt, InterviewQuestions)
            return data.questions
        except Exception:
            return ["Can you describe your technical experience?", "How do you handle challenging projects?"]

    def evaluate_interview(self, questions: List[str], answers: str) -> str:
        try:
            system_prompt = "You are an unbiased technical interviewer. Return only raw JSON matching the requested schema."
            user_prompt = f"""Evaluate the candidate's answers to the following questions.
            Questions: {questions}
            Answers: {answers}

            Return a JSON object matching this schema: {EvaluationResult.model_json_schema()}"""

            evaluation = self._call_llm(system_prompt, user_prompt, EvaluationResult)

            result_md = f"""### 📊 Evaluation Result: {evaluation.average_score}/10
**Decision: {evaluation.decision}**

**📈 Metrics:**
- Accuracy: {evaluation.accuracy_matching}%
- Consistency: {evaluation.consistency_score}%
- Bias Check: {evaluation.bias_check}

**Overall Feedback:**
{evaluation.overall_feedback}

**Question Breakdown:**"""
            for item in evaluation.question_scores:
                result_md += f"\n- **{item.question}**: Score {item.score}/10. {item.feedback}"

            return result_md
        except Exception as e:
            return f"❌ Error evaluating interview: {str(e)}"

## **Phase 6: Deployment**

In [ ]:
# Initialize Engine
rag_engine = RAGEngine(embeddings, reranker)
recruiter = RecruitmentAI(client, rag_engine)

# Premium Gradio Interface
CUSTOM_CSS = """
.gradio-container { font-family: 'Inter', -apple-system, sans-serif !important; }
.main-header {
    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
    padding: 2.5rem;
    border-radius: 1.5rem;
    color: white;
    text-align: center;
    margin-bottom: 2rem;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.1);
}
.main-header h1 { color: #f8fafc !important; margin-bottom: 0.5rem; font-weight: 800; }
.main-header p { color: #94a3b8; font-size: 1.1rem; }
"""

with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.HTML("""
        <div class='main-header'>
            <h1>💼 AI Recruitment Assistant</h1>
            <p>Advanced RAG-powered screening and interview evaluation system</p>
        </div>
    """)

    state_questions = gr.State([])
    state_score = gr.State(0)

    with gr.Tabs() as tabs:
        with gr.TabItem("📄 Stage 1: CV Screening", id=0):
            with gr.Row():
                with gr.Column(scale=1):
                    file_input = gr.File(label="Upload Candidate Resume (PDF)", file_types=[".pdf"])
                    job_input = gr.Textbox(label="Job Description", placeholder="Paste the job requirements here...", lines=10)
                    analyze_btn = gr.Button("🚀 Analyze Resume", variant="primary")
                with gr.Column(scale=1):
                    output_text = gr.Markdown("Analysis details will appear here.")
                    start_interview_btn = gr.Button("Proceed to Interview 🎤", variant="secondary", visible=False)

        with gr.TabItem("🎤 Stage 2: Interview", id=1):
            with gr.Column(visible=True) as prep_box:
                gr.Markdown("### ⚠️ Waiting for Screening\nPlease complete Stage 1 first.")
            with gr.Column(visible=False) as active_box:
                questions_display = gr.Markdown()
                answers_input = gr.Textbox(label="Candidate Answers", lines=12)
                eval_btn = gr.Button("📊 Submit for Evaluation", variant="primary")
                final_result = gr.Markdown()

    # Logic
    def on_analyze(file, job):
        if not file or not job: return "⚠️ Missing data.", [], 0, gr.update(visible=False)
        res, q, s = recruiter.analyze_resume(file.name, job)
        return res, q, s, gr.update(visible=s>=60)

    def on_start_interview(q):
        if not q: return gr.update(visible=True), gr.update(visible=False), ""
        ql = "\n".join([f"**Q{i+1}:** {x}" for i, x in enumerate(q)])
        return gr.update(visible=False), gr.update(visible=True), ql

    analyze_btn.click(on_analyze, [file_input, job_input], [output_text, state_questions, state_score, start_interview_btn])
    start_interview_btn.click(on_start_interview, [state_questions], [prep_box, active_box, questions_display])
    eval_btn.click(recruiter.evaluate_interview, [state_questions, answers_input], [final_result])

demo.launch(debug=True, share=True)

/tmp/ipykernel_2288/4078511788.py:21: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="blue")) as demo:
/tmp/ipykernel_2288/4078511788.py:21: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="blue")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cfd37ed87a785061fb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
